# Benchmark Mistral OCR 4 on single-page PDF → Markdown pairs

This notebook benchmarks Mistral OCR 4 with:

- `model = mistral-ocr-4-0`
- Markdown output from the OCR `pages[*].markdown` field
- inline Markdown tables by default
- `include_image_base64 = false`
- `image_limit = 0`
- `pages = [0]` for single-page PDFs
- chunked, recoverable execution

Expected input folder layout:

```text
/input_folder/
  sample_001.pdf
  sample_001.md
  sample_002.pdf
  sample_002.md
  ...
```

Each PDF is assumed to be a single-page document. The ground-truth Markdown file must have the same stem as its PDF.

Metrics included:

- Plain text: CER, WER, normalized edit similarity, character F1, word F1
- Markdown structure: AST/token-path F1 and reading-order LCS
- Tables: table count, table matching with TEDS-like tree similarity, cell exact F1, cell text similarity, row/column shape similarity
- Formulas: normalized LaTeX exact F1 and edit similarity


In [ ]:
# Colab setup
!pip -q install requests pandas tqdm rapidfuzz markdown-it-py beautifulsoup4 lxml apted scipy


In [ ]:
from google.colab import drive
# Comment this out if your data is not in Google Drive.
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import re
import json
import time
import math
import html
import base64
import shutil
import hashlib
import traceback
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein
from bs4 import BeautifulSoup
from markdown_it import MarkdownIt
from apted import APTED, Config
from scipy.optimize import linear_sum_assignment


## Configuration

Set `INPUT_DIR` to the folder containing `.pdf` + `.md` pairs.

For API keys, the notebook first reads `DATALAB_API_KEY` from the environment. If missing, it prompts securely with `getpass`.


In [ ]:
# ===== Paths =====
INPUT_DIR = Path('/content/drive/MyDrive/datasota/vitext')
RUN_DIR = Path('/content/drive/MyDrive/datasota/mistral_ocr4/vitext')

PRED_DIR = RUN_DIR / 'predictions_md'
RAW_JSON_DIR = RUN_DIR / 'raw_api_json'
ERROR_DIR = RUN_DIR / 'errors'
CHUNK_DIR = RUN_DIR / 'chunks'
METRIC_DIR = RUN_DIR / 'metrics'

for p in [RUN_DIR, PRED_DIR, RAW_JSON_DIR, ERROR_DIR, CHUNK_DIR, METRIC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ===== Mistral OCR API =====
MISTRAL_OCR_URL = 'https://api.mistral.ai/v1/ocr'

# Secure API key prompt if env var is absent.
try:
    from google.colab import userdata
    _mistral_api_key = userdata.get('MISTRAL_API_KEY')
except Exception:
    _mistral_api_key = None

if _mistral_api_key:
    os.environ['MISTRAL_API_KEY'] = _mistral_api_key

if not os.environ.get('MISTRAL_API_KEY'):
    import getpass
    os.environ['MISTRAL_API_KEY'] = getpass.getpass('MISTRAL_API_KEY: ')

API_KEY = os.environ['MISTRAL_API_KEY']

# ===== OCR options =====
MISTRAL_OCR_MODEL = 'mistral-ocr-4-0'

# The input PDFs are single-page files. Mistral page numbers are zero-based.
MAX_PAGES = 1

# Leave TABLE_FORMAT as None to keep tables inline in pages[*].markdown.
# Setting it to 'markdown' or 'html' returns tables separately and can introduce placeholders.
TABLE_FORMAT = None

INCLUDE_IMAGE_BASE64 = False
IMAGE_LIMIT = 0
INCLUDE_BLOCKS = False
CONFIDENCE_SCORES_GRANULARITY = None

# Mistral includes headers/footers in the main Markdown by default when these are False.
EXTRACT_HEADER = False
EXTRACT_FOOTER = False

# ===== Recoverable chunking =====
CHUNK_SIZE = 20
MAX_WORKERS = 1
START_CHUNK = 0          # Set to resume from a later chunk if desired.
RETRY_FAILED = True      # If True, failed docs are attempted again on rerun.
REQUEST_TIMEOUT = 600    # Mistral OCR is synchronous, so this covers the full request.

# ===== Metric normalization =====
LOWERCASE_FOR_METRICS = False
COLLAPSE_WHITESPACE = True


## Dataset discovery

This cell pairs each PDF with `stem.md` or `stem.markdown`. It writes `manifest.csv` so later cells have a stable benchmark manifest.


In [ ]:
def stable_id_from_path(path: Path) -> str:
    # Use stem unless duplicate stems exist; this keeps output filenames readable.
    return path.stem


def discover_pdf_md_pairs(input_dir: Path) -> pd.DataFrame:
    pdfs = sorted(input_dir.rglob('*.pdf'))
    rows = []
    missing = []

    for pdf_path in pdfs:
        md_path = pdf_path.with_suffix('.md')
        if not md_path.exists():
            md_path = pdf_path.with_suffix('.markdown')

        if md_path.exists():
            rows.append({
                'id': stable_id_from_path(pdf_path),
                'pdf_path': str(pdf_path),
                'gt_md_path': str(md_path),
                'pdf_name': pdf_path.name,
            })
        else:
            missing.append(str(pdf_path))

    df = pd.DataFrame(rows)
    if not df.empty and df['id'].duplicated().any():
        # Disambiguate duplicate stems by appending a short path hash.
        new_ids = []
        for _, row in df.iterrows():
            h = hashlib.sha1(row['pdf_path'].encode('utf-8')).hexdigest()[:8]
            new_ids.append(f"{Path(row['pdf_path']).stem}_{h}")
        df['id'] = new_ids

    manifest_path = RUN_DIR / 'manifest.csv'
    df.to_csv(manifest_path, index=False)

    print('Input dir:', input_dir)
    print('PDF/MD pairs:', len(df))
    print('PDFs missing MD ground truth:', len(missing))
    if missing[:10]:
        print('Examples missing GT:')
        for x in missing[:10]:
            print(' -', x)
    print('Manifest:', manifest_path)
    return df

manifest = discover_pdf_md_pairs(INPUT_DIR)
manifest.head()


Input dir: /content/drive/MyDrive/datasota/vitext
PDF/MD pairs: 23
PDFs missing MD ground truth: 0
Manifest: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/manifest.csv


,id,pdf_path,gt_md_path,pdf_name
0,vi_text_test1_page_001,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,vi_text_test1_page_001.pdf
1,vi_text_test1_page_002,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,vi_text_test1_page_002.pdf
2,vi_text_test1_page_003,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,vi_text_test1_page_003.pdf
3,vi_text_test1_page_004,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,vi_text_test1_page_004.pdf
4,vi_text_test1_page_005,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,vi_text_test1_page_005.pdf


## Mistral OCR 4 wrapper

The wrapper sends a base64-encoded PDF to Mistral OCR 4, saves the raw JSON response, and saves Markdown output separately. The per-document files are the first recoverability layer.


In [ ]:
def atomic_write_text(path: Path, text: str, encoding: str = 'utf-8') -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(text, encoding=encoding)
    tmp.replace(path)


def atomic_write_json(path: Path, obj: Any) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=2))


def encode_pdf_as_data_url(pdf_path: Path) -> str:
    with pdf_path.open('rb') as f:
        encoded = base64.b64encode(f.read()).decode('utf-8')
    return f'data:application/pdf;base64,{encoded}'


def pages_for_mistral(max_pages: Optional[int]) -> Optional[List[int]]:
    if max_pages is None:
        return None
    return list(range(int(max_pages)))


def make_ocr_payload(pdf_path: Path) -> Dict[str, Any]:
    payload = {
        'model': MISTRAL_OCR_MODEL,
        'document': {
            'type': 'document_url',
            'document_url': encode_pdf_as_data_url(pdf_path),
        },
        'pages': pages_for_mistral(MAX_PAGES),
        'include_image_base64': INCLUDE_IMAGE_BASE64,
        'image_limit': IMAGE_LIMIT,
        'include_blocks': INCLUDE_BLOCKS,
        'extract_header': EXTRACT_HEADER,
        'extract_footer': EXTRACT_FOOTER,
        'confidence_scores_granularity': CONFIDENCE_SCORES_GRANULARITY,
    }

    if TABLE_FORMAT is not None:
        payload['table_format'] = TABLE_FORMAT

    return {k: v for k, v in payload.items() if v is not None}


def extract_markdown_from_mistral_result(result: Dict[str, Any]) -> str:
    pages = result.get('pages') or []

    def page_index(page: Dict[str, Any]) -> int:
        try:
            return int(page.get('index', 0))
        except Exception:
            return 0

    markdown_pages = [
        str(page.get('markdown') or '')
        for page in sorted(pages, key=page_index)
    ]
    return '\n\n'.join(markdown_pages)


def convert_one_pdf(pdf_path: Path) -> Dict[str, Any]:
    headers = {
        'Authorization': f'Bearer {API_KEY}',
        'Content-Type': 'application/json',
    }
    payload = make_ocr_payload(pdf_path)

    r = requests.post(
        MISTRAL_OCR_URL,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )

    try:
        result = r.json()
    except Exception:
        raise RuntimeError(f'OCR returned non-JSON HTTP {r.status_code}: {r.text[:500]}')

    if not r.ok:
        raise RuntimeError(f'OCR failed HTTP {r.status_code}: {result}')

    result.setdefault('status', 'complete')
    result.setdefault('success', True)
    result['markdown'] = extract_markdown_from_mistral_result(result)
    result['page_count'] = len(result.get('pages') or [])
    return result


def process_manifest_row(row: pd.Series) -> Dict[str, Any]:
    sample_id = row['id']
    pdf_path = Path(row['pdf_path'])
    gt_path = Path(row['gt_md_path'])

    pred_path = PRED_DIR / f'{sample_id}.md'
    raw_path = RAW_JSON_DIR / f'{sample_id}.json'
    err_path = ERROR_DIR / f'{sample_id}.json'

    t0 = time.time()
    out = {
        'id': sample_id,
        'pdf_path': str(pdf_path),
        'gt_md_path': str(gt_path),
        'prediction_md_path': str(pred_path),
        'raw_json_path': str(raw_path),
        'status': 'unknown',
        'elapsed_seconds': None,
        'request_id': None,
        'page_count': None,
        'parse_quality_score': None,
        'error': None,
    }

    try:
        result = convert_one_pdf(pdf_path)
        out['elapsed_seconds'] = round(time.time() - t0, 3)
        out['request_id'] = result.get('id') or result.get('request_id')
        out['page_count'] = result.get('page_count')
        out['parse_quality_score'] = None

        atomic_write_json(raw_path, result)

        if result.get('status') == 'complete' and result.get('success', True):
            markdown = result.get('markdown')
            if markdown is None:
                markdown = ''
            atomic_write_text(pred_path, markdown)
            out['status'] = 'ok'
        else:
            out['status'] = 'failed'
            out['error'] = result.get('error') or json.dumps(result, ensure_ascii=False)[:1000]
            atomic_write_json(err_path, {'row': row.to_dict(), 'result': result, 'summary': out})

    except Exception as e:
        out['elapsed_seconds'] = round(time.time() - t0, 3)
        out['status'] = 'error'
        out['error'] = repr(e)
        atomic_write_json(err_path, {
            'row': row.to_dict(),
            'error': repr(e),
            'traceback': traceback.format_exc(),
            'summary': out,
        })

    return out


## Run OCR in chunks

Rerunning this cell skips documents that already have a saved Markdown prediction. Failed/error cases are retried if `RETRY_FAILED = True`.


In [ ]:
import time


def load_completed_ids() -> set:
    completed = set()
    for p in PRED_DIR.glob('*.md'):
        # Treat a saved Markdown file as completed. Empty Markdown is still a valid output.
        completed.add(p.stem)
    return completed


def write_chunk_csv_atomic(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    pd.DataFrame(rows).to_csv(tmp, index=False)
    tmp.replace(path)


if manifest.empty:
    raise ValueError('No PDF/MD pairs found. Check INPUT_DIR.')

all_records = []
num_chunks = math.ceil(len(manifest) / CHUNK_SIZE)

for chunk_idx in range(START_CHUNK, num_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(len(manifest), (chunk_idx + 1) * CHUNK_SIZE)
    chunk_df = manifest.iloc[start:end].copy()
    chunk_path = CHUNK_DIR / f'chunk_{start:06d}_{end-1:06d}.csv'

    completed_ids = load_completed_ids()
    pending_df = chunk_df[~chunk_df['id'].isin(completed_ids)].copy()

    print(f'\nChunk {chunk_idx + 1}/{num_chunks}: rows {start}..{end-1}')
    print(f'Already completed in this chunk: {len(chunk_df) - len(pending_df)}')
    print(f'Pending: {len(pending_df)}')

    chunk_records = []

    # Include lightweight records for skipped docs so the chunk CSV is self-contained.
    for _, row in chunk_df[chunk_df['id'].isin(completed_ids)].iterrows():
        sample_id = row['id']
        chunk_records.append({
            'id': sample_id,
            'pdf_path': row['pdf_path'],
            'gt_md_path': row['gt_md_path'],
            'prediction_md_path': str(PRED_DIR / f'{sample_id}.md'),
            'raw_json_path': str(RAW_JSON_DIR / f'{sample_id}.json'),
            'status': 'skipped_existing',
            'elapsed_seconds': 0,
            'request_id': None,
            'page_count': None,
            'parse_quality_score': None,
            'error': None,
        })

    if len(pending_df) > 0:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process_manifest_row, row) for _, row in pending_df.iterrows()]
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f'chunk {chunk_idx+1}'):
                rec = fut.result()
                chunk_records.append(rec)
                # Save partial chunk state after every completed document.
                write_chunk_csv_atomic(chunk_path, chunk_records)

    write_chunk_csv_atomic(chunk_path, chunk_records)
    all_records.extend(chunk_records)
    print('Wrote:', chunk_path)
    print("Wait for 30 seconds")
    time.sleep(15)  # Wait for 3 seconds

# Combined run summary from chunk files.
chunk_files = sorted(CHUNK_DIR.glob('chunk_*.csv'))
run_df = pd.concat([pd.read_csv(p) for p in chunk_files], ignore_index=True) if chunk_files else pd.DataFrame()
run_summary_path = RUN_DIR / 'run_summary.csv'
run_df.to_csv(run_summary_path, index=False)
print('\nRun summary:', run_summary_path)
run_df['status'].value_counts(dropna=False)



Chunk 1/2: rows 0..19
Already completed in this chunk: 20
Pending: 0
Wrote: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/chunks/chunk_000000_000019.csv
Wait for 30 seconds

Chunk 2/2: rows 20..22
Already completed in this chunk: 3
Pending: 0
Wrote: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/chunks/chunk_000020_000022.csv
Wait for 30 seconds

Run summary: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/run_summary.csv


,count
status,
skipped_existing,23


# Metric utilities

The metric layer reads saved Markdown predictions and ground truth. It does not call the API, so it is safe to rerun repeatedly.


In [ ]:
import re
import html
import math
import unicodedata
from typing import Any


def flatten_markdown_tables_for_text(s: str) -> str:
    """
    Flatten Markdown tables into plain OCR text.

    Examples:
        | A | B |        -> A B
        |---|---|        -> removed
        | 1. X | 7. Y |  -> 1. X 7. Y
        - | A | B |      -> A B

    This is for text-only CER/WER, not table-structure evaluation.
    """
    table_separator_re = re.compile(
        r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$"
    )

    out = []

    for line in str(s).splitlines():
        raw = line.strip()

        if not raw:
            continue

        # Remove accidental list marker before a table row:
        # "- | A | B |" -> "| A | B |"
        raw = re.sub(r"^\s*[-+*]\s+(?=\|)", "", raw)

        # Drop Markdown table separator rows:
        # |---|---|, |:---|---:|, | --- | :---: |
        if table_separator_re.fullmatch(raw):
            continue

        # Flatten rows that look like Markdown table rows.
        if "|" in raw:
            cells = raw.strip("|").split("|")

            cleaned_cells = []
            for cell in cells:
                cell = cell.strip()

                if not cell:
                    continue

                # Convert HTML line breaks inside cells.
                cell = re.sub(r"<\s*br\s*/?\s*>", " ", cell, flags=re.IGNORECASE)

                # Remove pure table/layout residue.
                if re.fullmatch(r":?-{3,}:?", cell):
                    continue

                cleaned_cells.append(cell)

            if cleaned_cells:
                out.append(" ".join(cleaned_cells))

            continue

        out.append(raw)

    return "\n".join(out)


def normalize_ocr_text(
    value: Any,
    *,
    lowercase: bool = False,
    ignore_soft_punctuation: bool = True,
) -> str:
    """
    Text-only OCR normalizer for Markdown/HTML OCR outputs.

    Designed for CER/WER on readable text content, not table structure.

    Key behavior:
    - Removes Markdown/HTML/layout syntax.
    - Flattens Markdown tables into text.
    - Removes image placeholders.
    - Preserves Vietnamese diacritics.
    - Preserves decimal commas: 0,1; 2703,81%.
    - Preserves thousands separators: 2.214.394.174.
    - Preserves literal footnote markers like (*).
    - Optionally ignores soft punctuation such as ':' and ';'.
    """

    if value is None:
        return ""

    try:
        if isinstance(value, float) and math.isnan(value):
            return ""
    except Exception:
        pass

    s = str(value)

    # ------------------------------------------------------------
    # 1. Basic cleanup
    # ------------------------------------------------------------
    s = html.unescape(s)
    s = unicodedata.normalize("NFC", s)

    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)
    s = s.replace("\u00a0", " ")
    s = s.replace("\u202f", " ")
    s = s.replace("\u00ad", "")
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # ------------------------------------------------------------
    # 2. Remove angle brackets around URLs/emails, then protect URLs
    # ------------------------------------------------------------
    s = re.sub(r"<(https?://[^>\s]+)>", r"\1", s)
    s = re.sub(
        r"<([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})>",
        r"\1",
        s,
    )

    protected = {}

    def protect(pattern: str, text: str, flags: int = 0) -> str:
        def repl(m: re.Match) -> str:
            key = f" PROTECTEDTOKEN{len(protected)} "
            protected[key.strip()] = m.group(0)
            return key

        return re.sub(pattern, repl, text, flags=flags)

    # Protect URLs before slash/dot normalization.
    s = protect(r"https?://[^\s)>\]]+", s)

    # ------------------------------------------------------------
    # 3. Remove images
    # ------------------------------------------------------------
    s = re.sub(r"<img\b[^>]*>", " ", s, flags=re.IGNORECASE)

    # Markdown images: remove placeholder entirely.
    s = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", s)

    # ------------------------------------------------------------
    # 4. HTML to text
    # ------------------------------------------------------------
    s = re.sub(r"<\s*br\s*/?\s*>", " ", s, flags=re.IGNORECASE)

    s = re.sub(
        r"</\s*(p|div|tr|li|h[1-6]|table|section)\s*>",
        " ",
        s,
        flags=re.IGNORECASE,
    )
    s = re.sub(
        r"<\s*(p|div|tr|li|h[1-6]|table|section)\b[^>]*>",
        " ",
        s,
        flags=re.IGNORECASE,
    )

    # Remove inline formatting tags but keep inner text.
    s = re.sub(
        r"</?\s*(b|strong|i|em|u|span|font|code|sup|sub)\b[^>]*>",
        "",
        s,
        flags=re.IGNORECASE,
    )

    # Remove any remaining HTML-like tags.
    s = re.sub(r"<[^>]+>", " ", s)

    # ------------------------------------------------------------
    # 5. Unescape Markdown escapes
    #    Important: (\*) -> (*)
    # ------------------------------------------------------------
    s = re.sub(r"\\([\\`*_{}\[\]()#+\-.!|])", r"\1", s)

    # Protect literal footnote markers before Markdown emphasis stripping.
    # Prevents (*) becoming ().
    s = protect(r"\(\s*\*+\s*\)", s)

    # ------------------------------------------------------------
    # 6. Markdown links, headings, blockquotes, bullets
    # ------------------------------------------------------------

    # [visible](url) -> visible
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)

    # Fenced code markers only; keep inner content.
    s = re.sub(r"```[A-Za-z0-9_-]*", " ", s)
    s = s.replace("```", " ")

    # Headings: ### Title -> Title
    s = re.sub(r"(?m)^\s{0,3}#{1,6}\s*", "", s)

    # Blockquotes.
    s = re.sub(r"(?m)^\s*>\s*", "", s)

    # Unordered list bullets.
    s = re.sub(r"(?m)^\s*[-*+]\s+(?=\S)", "", s)

    # Remove leftover standalone Markdown heading markers.
    # Handles cases like "#", "##", "###" that remain alone or inline.
    s = re.sub(r"(?m)^\s{0,3}#{1,6}\s*", "", s)
    s = re.sub(r"(?<!\S)#{1,6}(?!\S)", " ", s)

    # Remove leftover standalone Markdown emphasis/bullet markers.
    # Keeps literal footnote marker (*) protected earlier.
    s = re.sub(r"(?<!\S)[*_]{1,3}(?!\S)", " ", s)

    # ------------------------------------------------------------
    # 7. Flatten Markdown tables
    # ------------------------------------------------------------
    s = flatten_markdown_tables_for_text(s)

    # ------------------------------------------------------------
    # 8. Markdown emphasis stripping
    # ------------------------------------------------------------

    # Bold markers.
    s = re.sub(r"(\*\*|__)(.*?)\1", r"\2", s)

    # Single emphasis.
    # This is intentionally conservative to avoid eating across footnotes.
    s = re.sub(r"(?<!\S)\*([^\s*][^*\n]{0,120}?[^\s*])\*(?!\S)", r"\1", s)
    s = re.sub(r"(?<!\S)_([^\s_][^_\n]{0,120}?[^\s_])_(?!\S)", r"\1", s)

    # Inline code.
    s = re.sub(r"`([^`]*)`", r"\1", s)

    # Remove leftover Markdown formatting artifacts.
    # Remove leftover Markdown formatting artifacts.
    s = re.sub(r"[*_]{1,3}", " ", s)
    s = re.sub(r"(?<!\S)#{1,6}(?!\S)", " ", s)

    # Normalize bullet/list glyph variants.
    s = re.sub(r"[■□▪▫●○◦•∙·]", "", s)

    # ------------------------------------------------------------
    # 9. Normalize layout separators
    # ------------------------------------------------------------
    s = s.replace("•", " ")
    s = s.replace("–", "-")
    s = s.replace("—", "-")
    s = s.replace("／", "/")
    s = s.replace("·", " ")
    s = s.replace("∙", " ")
    s = s.replace("●", " ")
    s = s.replace("•", " ")

    # ------------------------------------------------------------
    # 10. Number-aware punctuation spacing
    # ------------------------------------------------------------

    # Preserve decimal commas.
    s = re.sub(r"(?<=\d),\s+(?=\d)", ",", s)

    # Preserve thousands-dot numbers.
    s = re.sub(r"(?<=\d)\.\s+(?=\d)", ".", s)

    # Fix spaces before percent.
    s = re.sub(r"\s+%", "%", s)

    # Optional: ignore weak punctuation separators for text-only OCR.
    # This makes these equivalent:
    # "Hành khách; triệu khách"
    # "Hành khách: triệu khách"
    # "Ghế suất:%"
    if ignore_soft_punctuation:
        s = re.sub(r"(?<!\d)\s*[:;]\s*(?!\d)", " ", s)
    else:
        s = re.sub(r"\s*;\s*", " ; ", s)
        s = re.sub(r"\s*:\s*", ": ", s)
        s = re.sub(r"(?<=\d):\s+(?=\d)", ":", s)

    # Commas in prose, but not decimal commas.
    s = re.sub(r"(?<!\d),\s*(?!\d)", ", ", s)

    # Remove spaces before punctuation.
    s = re.sub(r"\s+([,.;:%])", r"\1", s)

    # Parentheses spacing.
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)

    # ------------------------------------------------------------
    # 11. Restore protected tokens
    # ------------------------------------------------------------
    for key, original in protected.items():
        s = s.replace(key, original)

    # Final whitespace.
    s = re.sub(r"\s+", " ", s).strip()

    # Remove runs of 3+ ASCII dots, optionally separated by spaces.
    # Examples: ".....", ". . . . .", "... ..."
    s = re.sub(r"(?:\.\s*){3,}", " ", s)

    if lowercase:
        s = s.lower()

    return s

In [ ]:
# Markdown renderer/parser for text and AST-like token metrics.
MD = MarkdownIt('commonmark').enable('table')


def read_text(path: Path) -> str:
    return path.read_text(encoding='utf-8', errors='replace')


def normalize_ws(s: str) -> str:
    s = html.unescape(str(s or ''))
    if COLLAPSE_WHITESPACE:
        s = re.sub(r'\s+', ' ', s).strip()
    if LOWERCASE_FOR_METRICS:
        s = s.lower()
    return s

def normalize(s: str) -> str:
    """
    Normalize table-cell content while preserving meaningful inline Markdown formatting.

    Converts:
    - HTML italic tags to Markdown italic: <i>x</i>, <em>x</em> -> *x*
    - HTML bold tags to Markdown bold: <b>x</b>, <strong>x</strong> -> **x**
    - HTML code tags to Markdown code: <code>x</code> -> `x`
    - HTML entities to literal characters
    - escaped Markdown table pipes: \\| -> |

    Also normalizes whitespace.
    """
    s = html.unescape(str(s or ""))

    # URLs inside angle brackets
    s = re.sub(r'<((?:https?://|www\.)[^>\s]+)>', r'\1', s)

    # Email autolinks inside angle brackets
    s = re.sub(r'<([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})>', r'\1', s)

    # Convert HTML inline formatting to canonical Markdown.
    s = re.sub(
        r"<\s*(i|em)\s*>(.*?)<\s*/\s*\1\s*>",
        r"*\2*",
        s,
        flags=re.IGNORECASE | re.DOTALL,
    )

    s = re.sub(
        r"<\s*(b|strong)\s*>(.*?)<\s*/\s*\1\s*>",
        r"**\2**",
        s,
        flags=re.IGNORECASE | re.DOTALL,
    )

    s = re.sub(
        r"<\s*code\s*>(.*?)<\s*/\s*code\s*>",
        r"`\1`",
        s,
        flags=re.IGNORECASE | re.DOTALL,
    )

    # Remove remaining HTML tags but keep their inner text.
    s = re.sub(r"<[^>]+>", " ", s)

    # Normalize equivalent Markdown emphasis forms.
    # __bold__ -> **bold**
    s = re.sub(r"__(.*?)__", r"**\1**", s, flags=re.DOTALL)

    # _italic_ -> *italic*
    # This is intentionally conservative to avoid breaking underscores inside words.
    s = re.sub(r"(?<!\w)_(?!_)(.*?)(?<!_)_(?!\w)", r"*\1*", s, flags=re.DOTALL)

    # Unescape table pipes.
    s = re.sub(r"\\([|])", r"\1", s)

    #normalize header
    s = re.sub(r'(?<=\|)\s*:?-{3,}:?\s*(?=\|)', '---', s)

    s = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', s)

    return normalize_ws(s)
    #return s

def normalize_cell(s: str) -> str:
    s = html.unescape(str(s or ''))
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'\*\*|__|\*|_|`', '', s)
    s = re.sub(r'\\([|])', r'\1', s)
    return normalize_ws(s)


def edit_distance(a: str, b: str) -> int:
    return Levenshtein.distance(a or '', b or '')


def normalized_edit_similarity(ref: str, pred: str) -> float:
    ref = ref or ''
    pred = pred or ''
    denom = max(len(ref), len(pred), 1)
    return max(0.0, 1.0 - edit_distance(ref, pred) / denom)


def cer(ref: str, pred: str) -> float:
    ref = ref or ''
    pred = pred or ''
    return edit_distance(ref, pred) / max(len(ref), 1)


def tokenize_words(s: str) -> List[str]:
    return re.findall(r'\w+|[^\w\s]', s, flags=re.UNICODE)


def wer(ref: str, pred: str) -> float:
    r = tokenize_words(ref)
    p = tokenize_words(pred)
    return Levenshtein.distance(r, p) / max(len(r), 1)


def multiset_prf(ref_items: List[Any], pred_items: List[Any]) -> Dict[str, float]:
    ref_c = Counter(ref_items)
    pred_c = Counter(pred_items)
    overlap = sum((ref_c & pred_c).values())
    pred_n = sum(pred_c.values())
    ref_n = sum(ref_c.values())
    precision = overlap / pred_n if pred_n else (1.0 if ref_n == 0 else 0.0)
    recall = overlap / ref_n if ref_n else (1.0 if pred_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1}


def char_f1(ref: str, pred: str) -> float:
    return multiset_prf(list(ref), list(pred))['f1']


def word_f1(ref: str, pred: str) -> float:
    return multiset_prf(tokenize_words(ref), tokenize_words(pred))['f1']


def markdown_to_plain_text(md: str) -> str:
    # Render Markdown to HTML, then extract visible text. Fall back to regex stripping if rendering fails.
    try:
        rendered = MD.render(md or '')
        soup = BeautifulSoup(rendered, 'lxml')
        text = soup.get_text('\n')
    except Exception:
        text = md or ''
        text = re.sub(r'!\[([^\]]*)\]\([^)]*\)', r'\1', text)
        text = re.sub(r'\[([^\]]+)\]\([^)]*\)', r'\1', text)
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'[`*_>#~-]+', ' ', text)
    return normalize_ws(text)


def extract_blocks_for_order(md: str) -> List[str]:
    # Coarse reading-order metric: sequence of non-empty visible text blocks.
    #text = markdown_to_plain_text(md)
    text = normalize(md)
    blocks = [normalize_ws(x) for x in re.split(r'\n+|(?<=[.!?。！？])\s+', text) if normalize_ws(x)]
    return blocks


def lcs_len(a: List[Any], b: List[Any]) -> int:
    # Memory-efficient LCS for moderate lists.
    if len(a) < len(b):
        a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j-1] + 1 if x == y else max(prev[j], cur[-1]))
        prev = cur
    return prev[-1]


def reading_order_lcs(ref_md: str, pred_md: str) -> float:
    ref_blocks = extract_blocks_for_order(ref_md)
    pred_blocks = extract_blocks_for_order(pred_md)
    if not ref_blocks and not pred_blocks:
        return 1.0
    # Exact block LCS is strict. This is useful as a low-noise order indicator.
    return lcs_len(ref_blocks, pred_blocks) / max(len(ref_blocks), 1)


## Markdown AST / structure metrics

This is intentionally not the main table metric. It captures whether headings, lists, paragraphs, code blocks, and table tokens appear in a similar hierarchy/order.


In [ ]:
def markdown_token_paths(md: str, include_text: bool = False) -> List[str]:
    tokens = MD.parse(md or '')
    stack = []
    paths = []

    for tok in tokens:
        t = tok.type
        if tok.nesting == 1:
            stack.append(t)
            paths.append('/'.join(stack))
        elif tok.nesting == -1:
            paths.append('/'.join(stack + [t]))
            if stack:
                stack.pop()
        else:
            path = '/'.join(stack + [t])
            paths.append(path)
            if include_text and tok.content and t in {'inline', 'text', 'code_inline', 'fence', 'code_block'}:
                content = normalize_ws(tok.content)
                if content:
                    paths.append(path + '::TEXT::' + content)
    return paths


def markdown_structure_metrics(ref_md: str, pred_md: str) -> Dict[str, float]:
    ref_struct = markdown_token_paths(ref_md, include_text=False)
    pred_struct = markdown_token_paths(pred_md, include_text=False)
    struct = multiset_prf(ref_struct, pred_struct)

    ref_struct_text = markdown_token_paths(ref_md, include_text=True)
    pred_struct_text = markdown_token_paths(pred_md, include_text=True)
    struct_text = multiset_prf(ref_struct_text, pred_struct_text)

    return {
        'ast_struct_precision': struct['precision'],
        'ast_struct_recall': struct['recall'],
        'ast_struct_f1': struct['f1'],
        'ast_struct_text_precision': struct_text['precision'],
        'ast_struct_text_recall': struct_text['recall'],
        'ast_struct_text_f1': struct_text['f1'],
        'reading_order_lcs': reading_order_lcs(ref_md, pred_md),
    }


## Table extraction and table metrics

Tables need separate metrics because a Markdown AST score can look acceptable even when the table grid is wrong. This notebook extracts both pipe-style Markdown tables and inline HTML `<table>` blocks.

The TEDS-like metric below uses tree edit distance on a canonical HTML table tree, with edit cost softened by cell text similarity.


In [ ]:
def is_pipe_table_separator(line: str) -> bool:
    s = line.strip()
    if '|' not in s:
        return False
    s2 = s.strip('|').strip()
    if not s2:
        return False
    parts = [p.strip() for p in s2.split('|')]
    return all(re.fullmatch(r':?-{3,}:?', p or '') is not None for p in parts)


def looks_like_pipe_row(line: str) -> bool:
    s = line.strip()
    return '|' in s and bool(s.strip('|').strip())


def split_pipe_row(line: str) -> List[str]:
    s = line.strip()
    if s.startswith('|'):
        s = s[1:]
    if s.endswith('|'):
        s = s[:-1]

    cells = []
    cur = []
    escaped = False
    for ch in s:
        if escaped:
            cur.append(ch)
            escaped = False
        elif ch == '\\':
            escaped = True
        elif ch == '|':
            cells.append(''.join(cur))
            cur = []
        else:
            cur.append(ch)
    cells.append(''.join(cur))
    return [normalize_cell(c) for c in cells]


def rectangularize(rows: List[List[str]]) -> List[List[str]]:
    if not rows:
        return []
    width = max(len(r) for r in rows)
    return [r + [''] * (width - len(r)) for r in rows]


def extract_pipe_tables(md: str) -> List[List[List[str]]]:
    lines = (md or '').splitlines()
    tables = []
    i = 0
    while i < len(lines) - 1:
        if looks_like_pipe_row(lines[i]) and is_pipe_table_separator(lines[i + 1]):
            rows = [split_pipe_row(lines[i])]
            i += 2
            while i < len(lines) and looks_like_pipe_row(lines[i]):
                # Stop if the row is likely not table content.
                rows.append(split_pipe_row(lines[i]))
                i += 1
            tables.append(rectangularize(rows))
        else:
            i += 1
    return tables


def extract_html_tables(md: str) -> List[List[List[str]]]:
    soup = BeautifulSoup(md or '', 'lxml')
    out = []
    for table in soup.find_all('table'):
        rows = []
        for tr in table.find_all('tr'):
            cells = []
            for cell in tr.find_all(['th', 'td']):
                # Basic handling: rowspans/colspans are ignored in the grid metric but retained in TEDS if using raw HTML.
                cells.append(normalize_cell(cell.get_text(' ')))
            if cells:
                rows.append(cells)
        if rows:
            out.append(rectangularize(rows))
    return out


def extract_tables(md: str) -> List[List[List[str]]]:
    # If HTML tables exist, include them. Pipe tables are also included.
    return extract_pipe_tables(md) + extract_html_tables(md)


def table_to_html_grid(table: List[List[str]]) -> str:
    rows = []
    for r, row in enumerate(table):
        tag = 'th' if r == 0 else 'td'
        row_html = ''.join(f'<{tag}>{html.escape(str(c))}</{tag}>' for c in row)
        rows.append(f'<tr>{row_html}</tr>')
    return '<table>' + ''.join(rows) + '</table>'


class TreeNode:
    def __init__(self, label: str, children: Optional[List['TreeNode']] = None):
        self.label = label
        self.children = children or []


def soup_node_to_tree(node) -> Optional[TreeNode]:
    if getattr(node, 'name', None) is None:
        text = normalize_cell(str(node))
        if text:
            return TreeNode('text:' + text, [])
        return None

    children = []
    for child in getattr(node, 'children', []):
        t = soup_node_to_tree(child)
        if t is not None:
            children.append(t)

    if node.name in {'td', 'th'}:
        # Store all cell text in the cell node label. This makes content similarity part of rename cost.
        label = f'{node.name}:{normalize_cell(node.get_text(" "))}'
        return TreeNode(label, [])

    return TreeNode(str(node.name), children)


def html_table_to_tree(html_table: str) -> TreeNode:
    soup = BeautifulSoup(html_table, 'lxml')
    table = soup.find('table')
    if table is None:
        return TreeNode('table', [])
    return soup_node_to_tree(table) or TreeNode('table', [])


def count_tree_nodes(node: TreeNode) -> int:
    return 1 + sum(count_tree_nodes(c) for c in node.children)


class TEDSConfig(Config):
    def children(self, node):
        return node.children

    def rename(self, n1, n2):
        l1 = n1.label
        l2 = n2.label
        tag1 = l1.split(':', 1)[0]
        tag2 = l2.split(':', 1)[0]
        if tag1 != tag2:
            return 1.0
        if tag1 in {'td', 'th', 'text'}:
            text1 = l1.split(':', 1)[1] if ':' in l1 else ''
            text2 = l2.split(':', 1)[1] if ':' in l2 else ''
            return 1.0 - normalized_edit_similarity(text1, text2)
        return 0.0

    def insert(self, node):
        return 1.0

    def delete(self, node):
        return 1.0


def teds_similarity_table(ref_table: List[List[str]], pred_table: List[List[str]]) -> float:
    ref_tree = html_table_to_tree(table_to_html_grid(ref_table))
    pred_tree = html_table_to_tree(table_to_html_grid(pred_table))
    dist = APTED(ref_tree, pred_tree, TEDSConfig()).compute_edit_distance()
    denom = max(count_tree_nodes(ref_tree), count_tree_nodes(pred_tree), 1)
    return max(0.0, 1.0 - dist / denom)


def cell_exact_f1_aligned(ref_table: List[List[str]], pred_table: List[List[str]]) -> float:
    ref_items = []
    pred_items = []
    for r, row in enumerate(ref_table):
        for c, val in enumerate(row):
            ref_items.append((r, c, normalize_cell(val)))
    for r, row in enumerate(pred_table):
        for c, val in enumerate(row):
            pred_items.append((r, c, normalize_cell(val)))
    return multiset_prf(ref_items, pred_items)['f1']


def avg_cell_text_similarity_aligned(ref_table: List[List[str]], pred_table: List[List[str]]) -> float:
    ref_rows = len(ref_table)
    pred_rows = len(pred_table)
    ref_cols = max([len(r) for r in ref_table], default=0)
    pred_cols = max([len(r) for r in pred_table], default=0)
    rows = max(ref_rows, pred_rows)
    cols = max(ref_cols, pred_cols)
    if rows == 0 or cols == 0:
        return 1.0 if rows == 0 and cols == 0 else 0.0

    sims = []
    for r in range(rows):
        for c in range(cols):
            rv = ref_table[r][c] if r < ref_rows and c < len(ref_table[r]) else ''
            pv = pred_table[r][c] if r < pred_rows and c < len(pred_table[r]) else ''
            sims.append(normalized_edit_similarity(normalize_cell(rv), normalize_cell(pv)))
    return float(np.mean(sims)) if sims else 1.0


def table_shape_similarity(ref_table: List[List[str]], pred_table: List[List[str]]) -> Dict[str, float]:
    rr = len(ref_table)
    pr = len(pred_table)
    rc = max([len(r) for r in ref_table], default=0)
    pc = max([len(r) for r in pred_table], default=0)
    row_sim = 1.0 - abs(rr - pr) / max(rr, pr, 1)
    col_sim = 1.0 - abs(rc - pc) / max(rc, pc, 1)
    return {'row_count_similarity': row_sim, 'col_count_similarity': col_sim}


def match_tables(ref_tables: List[List[List[str]]], pred_tables: List[List[List[str]]]) -> Tuple[List[Tuple[int, int, float]], float]:
    if not ref_tables and not pred_tables:
        return [], 1.0
    if not ref_tables or not pred_tables:
        return [], 0.0

    sim = np.zeros((len(ref_tables), len(pred_tables)), dtype=float)
    for i, rt in enumerate(ref_tables):
        for j, pt in enumerate(pred_tables):
            sim[i, j] = teds_similarity_table(rt, pt)

    row_ind, col_ind = linear_sum_assignment(1.0 - sim)
    matches = [(int(i), int(j), float(sim[i, j])) for i, j in zip(row_ind, col_ind)]
    # Penalize missing or extra tables by normalizing over max count.
    corpus_like_score = sum(s for _, _, s in matches) / max(len(ref_tables), len(pred_tables), 1)
    return matches, corpus_like_score


def table_metrics(ref_md: str, pred_md: str) -> Dict[str, float]:
    ref_tables = extract_tables(ref_md)
    pred_tables = extract_tables(pred_md)
    matches, teds_doc = match_tables(ref_tables, pred_tables)

    matched_teds = []
    matched_cell_f1 = []
    matched_cell_sim = []
    matched_row_sim = []
    matched_col_sim = []

    for i, j, sim in matches:
        rt = ref_tables[i]
        pt = pred_tables[j]
        matched_teds.append(sim)
        matched_cell_f1.append(cell_exact_f1_aligned(rt, pt))
        matched_cell_sim.append(avg_cell_text_similarity_aligned(rt, pt))
        shape = table_shape_similarity(rt, pt)
        matched_row_sim.append(shape['row_count_similarity'])
        matched_col_sim.append(shape['col_count_similarity'])

    table_count_precision = min(len(ref_tables), len(pred_tables)) / max(len(pred_tables), 1) if pred_tables else (1.0 if not ref_tables else 0.0)
    table_count_recall = min(len(ref_tables), len(pred_tables)) / max(len(ref_tables), 1) if ref_tables else (1.0 if not pred_tables else 0.0)
    table_count_f1 = (2 * table_count_precision * table_count_recall / (table_count_precision + table_count_recall)
                      if table_count_precision + table_count_recall else 0.0)

    return {
        'ref_table_count': len(ref_tables),
        'pred_table_count': len(pred_tables),
        'table_count_precision': table_count_precision,
        'table_count_recall': table_count_recall,
        'table_count_f1': table_count_f1,
        'table_teds_doc': teds_doc,
        'table_teds_matched_mean': float(np.mean(matched_teds)) if matched_teds else (1.0 if not ref_tables and not pred_tables else 0.0),
        'table_cell_exact_f1_mean': float(np.mean(matched_cell_f1)) if matched_cell_f1 else (1.0 if not ref_tables and not pred_tables else 0.0),
        'table_cell_text_similarity_mean': float(np.mean(matched_cell_sim)) if matched_cell_sim else (1.0 if not ref_tables and not pred_tables else 0.0),
        'table_row_count_similarity_mean': float(np.mean(matched_row_sim)) if matched_row_sim else (1.0 if not ref_tables and not pred_tables else 0.0),
        'table_col_count_similarity_mean': float(np.mean(matched_col_sim)) if matched_col_sim else (1.0 if not ref_tables and not pred_tables else 0.0),
    }


## Formula metrics

This is a lightweight normalized-LaTeX metric. Use it to catch formula omissions and obvious OCR/formatting corruption. For serious math benchmarking, add a domain-specific LaTeX normalizer or symbolic equivalence checker.


In [ ]:
FORMULA_PATTERNS = [
    re.compile(r'\$\$(.+?)\$\$', re.DOTALL),
    re.compile(r'(?<!\\)\$(?!\$)(.+?)(?<!\\)\$', re.DOTALL),
    re.compile(r'\\\[(.+?)\\\]', re.DOTALL),
    re.compile(r'\\\((.+?)\\\)', re.DOTALL),
]


def extract_formulas(md: str) -> List[str]:
    formulas = []
    text = md or ''
    for pat in FORMULA_PATTERNS:
        formulas.extend([m.group(1) for m in pat.finditer(text)])
    return [normalize_latex(x) for x in formulas if normalize_latex(x)]


def normalize_latex(s: str) -> str:
    s = html.unescape(str(s or ''))
    s = s.strip()
    s = re.sub(r'\\left|\\right', '', s)
    s = re.sub(r'\s+', '', s)
    return s


def formula_metrics(ref_md: str, pred_md: str) -> Dict[str, float]:
    ref = extract_formulas(ref_md)
    pred = extract_formulas(pred_md)
    exact = multiset_prf(ref, pred)

    if not ref and not pred:
        mean_best_sim = 1.0
    elif not ref or not pred:
        mean_best_sim = 0.0
    else:
        sims = []
        for rf in ref:
            sims.append(max(normalized_edit_similarity(rf, pf) for pf in pred))
        mean_best_sim = float(np.mean(sims))

    return {
        'ref_formula_count': len(ref),
        'pred_formula_count': len(pred),
        'formula_exact_precision': exact['precision'],
        'formula_exact_recall': exact['recall'],
        'formula_exact_f1': exact['f1'],
        'formula_best_edit_similarity_mean': mean_best_sim,
    }


# Compute document-level metrics

This cell joins the manifest with saved predictions and produces one row per document.


In [ ]:
def compute_metrics_for_pair(ref_md: str, pred_md: str) -> Dict[str, float]:
    ref_text = normalize_ocr_text(ref_md)
    pred_text = normalize_ocr_text(pred_md)
    #ref_text = markdown_to_plain_text(ref_md)
    #pred_text = markdown_to_plain_text(pred_md)

    out = {
        'ref_text_len': len(ref_text),
        'pred_text_len': len(pred_text),
        'cer': cer(ref_text, pred_text),
        'wer': wer(ref_text, pred_text),
        'normalized_edit_similarity': normalized_edit_similarity(ref_text, pred_text),
        'char_f1': char_f1(ref_text, pred_text),
        'word_f1': word_f1(ref_text, pred_text),
    }
    out.update(markdown_structure_metrics(ref_md, pred_md))
    out.update(table_metrics(ref_md, pred_md))
    out.update(formula_metrics(ref_md, pred_md))
    return out


metric_rows = []
for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='metrics'):
    sample_id = row['id']
    gt_path = Path(row['gt_md_path'])
    pred_path = PRED_DIR / f'{sample_id}.md'

    rec = {
        'id': sample_id,
        'pdf_path': row['pdf_path'],
        'gt_md_path': row['gt_md_path'],
        'prediction_md_path': str(pred_path),
        'metric_status': 'unknown',
        'metric_error': None,
    }

    try:
        if not pred_path.exists():
            rec['metric_status'] = 'missing_prediction'
        else:
            ref_md = read_text(gt_path)
            pred_md = read_text(pred_path)
            rec.update(compute_metrics_for_pair(ref_md, pred_md))
            rec['metric_status'] = 'ok'
    except Exception as e:
        rec['metric_status'] = 'error'
        rec['metric_error'] = repr(e)

    metric_rows.append(rec)

metrics_df = pd.DataFrame(metric_rows)
metrics_path = METRIC_DIR / 'document_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print('Saved:', metrics_path)
metrics_df['metric_status'].value_counts(dropna=False)


metrics:   0%|          | 0/23 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/metrics/document_metrics.csv


,count
metric_status,
ok,23


# Aggregate summary

This cell reports macro averages across documents and subset averages for documents containing tables/formulas.


In [ ]:
ok = metrics_df[metrics_df['metric_status'] == 'ok'].copy()

metric_cols = [
    'cer', 'wer', 'normalized_edit_similarity', 'char_f1', 'word_f1',
    'ast_struct_f1', 'ast_struct_text_f1', 'reading_order_lcs',
    'table_count_f1', 'table_teds_doc', 'table_teds_matched_mean',
    'table_cell_exact_f1_mean', 'table_cell_text_similarity_mean',
    'table_row_count_similarity_mean', 'table_col_count_similarity_mean',
    'formula_exact_f1', 'formula_best_edit_similarity_mean',
]

summary_rows = []

def add_summary(name: str, df: pd.DataFrame):
    row = {'subset': name, 'n_docs': len(df)}
    for c in metric_cols:
        if c in df.columns:
            row[c + '_mean'] = pd.to_numeric(df[c], errors='coerce').mean()
            row[c + '_median'] = pd.to_numeric(df[c], errors='coerce').median()
    summary_rows.append(row)

add_summary('all_ok_docs', ok)
if 'ref_table_count' in ok.columns:
    add_summary('docs_with_gt_tables', ok[ok['ref_table_count'].fillna(0) > 0])
if 'ref_formula_count' in ok.columns:
    add_summary('docs_with_gt_formulas', ok[ok['ref_formula_count'].fillna(0) > 0])

summary_df = pd.DataFrame(summary_rows)
summary_path = METRIC_DIR / 'summary_metrics.csv'
summary_df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary_df.T


Saved: /content/drive/MyDrive/datasota/mistral_ocr4/vitext/metrics/summary_metrics.csv


,0,1,2
subset,all_ok_docs,docs_with_gt_tables,docs_with_gt_formulas
n_docs,23,12,13
cer_mean,0.061959,0.057196,0.068858
cer_median,0.005219,0.011064,0.013401
wer_mean,0.073451,0.06729,0.08433
wer_median,0.010043,0.012796,0.015228
normalized_edit_similarity_mean,0.938045,0.942811,0.931149
normalized_edit_similarity_median,0.994781,0.988936,0.986599
char_f1_mean,0.960504,0.967122,0.957724
char_f1_median,0.996695,0.997173,0.995431


# Error analysis helpers

Use these cells to inspect worst examples by each metric and compare ground truth vs prediction quickly.


In [ ]:
def show_worst(metric: str, ascending: bool = True, n: int = 10):
    cols = ['id', metric, 'pdf_path', 'gt_md_path', 'prediction_md_path']
    cols = [c for c in cols if c in ok.columns]
    return ok.sort_values(metric, ascending=ascending)[cols].head(n)

# Examples:
# Low TEDS means table structure/cell content mismatched.
show_worst('table_teds_doc', ascending=True, n=10)


,id,table_teds_doc,pdf_path,gt_md_path,prediction_md_path
3,vi_text_test1_page_004,0.000000,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
12,vi_text_test2_page_003,0.000000,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
17,vi_text_test3_page_003,0.493391,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
6,vi_text_test1_page_007,0.500000,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
1,vi_text_test1_page_002,0.934620,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
22,vi_text_test3_page_008,0.981609,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
4,vi_text_test1_page_005,0.989899,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
7,vi_text_test1_page_008,0.995034,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
9,vi_text_test1_page_010,0.995402,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...
2,vi_text_test1_page_003,1.000000,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/vitext/vi_text...,/content/drive/MyDrive/datasota/mistral_ocr4/v...


In [ ]:
def print_pair(sample_id: str, max_chars: int = 4000):
    row = manifest[manifest['id'] == sample_id].iloc[0]
    ref = read_text(Path(row['gt_md_path']))
    pred = read_text(PRED_DIR / f'{sample_id}.md')
    print('=' * 80)
    print('GROUND TRUTH')
    print('=' * 80)
    print(ref[:max_chars])
    print('\n' + '=' * 80)
    print('PREDICTION')
    print('=' * 80)
    print(pred[:max_chars])

# Example:
print_pair(ok.sort_values('table_teds_doc').iloc[2]['id'])


GROUND TRUTH
## BÀI TẬP CUỐI CHƯƠNG V

1. Người ta tiến hành phỏng vấn 40 người về một mẫu áo sơ mi mới. Người điều tra yêu cầu cho điểm mẫu áo đó theo thang điểm là 100. Kết quả được trình bày trong *Bảng 16*.

|  Nhóm | Tần số | Tần số tích lũy  |
| --- | --- | --- |
|  [50 ; 60) | 4 | 4  |
|  [60 ; 70) | 5 | 9  |
|  [70 ; 80) | 23 | 32  |
|  [80 ; 90) | 6 | 38  |
|  [90 ; 100) | 2 | 40  |
|   | $n = 40$ |   |

Bảng 16

- a) Trung vị của mẫu số liệu ghép nhóm trên gần nhất với giá trị:

A. 74. B. 75. C. 76. D. 77.

- b) Tứ phân vị của mẫu số liệu ghép nhóm trên (làm tròn kết quả đến hàng đơn vị) là:

A. $Q_1 \approx 71$; $Q_2 \approx 76$; $Q_3 \approx 78$. B. $Q_1 \approx 71$; $Q_2 \approx 75$; $Q_3 \approx 78$.
C. $Q_1 \approx 70$; $Q_2 \approx 76$; $Q_3 \approx 79$. D. $Q_1 \approx 70$; $Q_2 \approx 75$; $Q_3 \approx 79$.

- c) Mốt của mẫu số liệu ghép nhóm trên (làm tròn kết quả đến hàng đơn vị) là:

A. 73. B. 74. C. 75. D. 76.

2. Chọn ngẫu nhiên hai số khác nhau từ 21 số nguyên 

In [ ]:
def print_normalized_pair(sample_id: str, max_chars: int = 4000):
    row = manifest[manifest['id'] == sample_id].iloc[0]
    ref = read_text(Path(row['gt_md_path']))
    pred = read_text(PRED_DIR / f'{sample_id}.md')

    # Normalize the prediction as requested
    norm_pred = normalize_ocr_text(pred)
    norm_ref = normalize_ocr_text(ref)
    print('=' * 80)
    print(f'SAMPLE ID: {sample_id}')
    print('=' * 80)
    print('GROUND TRUTH')
    print('=' * 80)
    print(ref[:max_chars])
    print('\n' + '=' * 80)
    print('PREDICTION')
    print('=' * 80)
    print(pred[:max_chars])

    print('=' * 80)
    print(f'SAMPLE ID: {sample_id}')
    print('=' * 80)
    print('NORMALIZED GROUND TRUTH')
    print('=' * 80)
    print(norm_ref[:max_chars])
    print('\n' + '=' * 80)
    print('NORMALIZED PREDICTION')
    print('=' * 80)
    print(norm_pred[:max_chars])

# Identify the worst example by ast_struct_f1
worst_id = ok.sort_values('cer', ascending=False).iloc[0]['id']
print_normalized_pair(worst_id)
ok[ok['id'] == worst_id]['cer']

SAMPLE ID: vi_text_test3_page_001
GROUND TRUTH
**8** Một hộp có 5 viên bi màu xanh, 6 viên bi màu đỏ và 7 viên bi màu vàng. Chọn ngẫu nhiên 5 viên bi trong hộp. Tính xác suất để 5 viên bi được chọn có đủ ba màu và số bi màu đỏ bằng số bi màu vàng.

## *Giải*

- Mỗi cách chọn ra đồng thời 3 học sinh trong câu lạc bộ cho ta một tổ hợp chập 3 của 15 phần tử. Do đó, không gian mẫu $\Omega$ gồm các tổ hợp chập 3 của 15 phần tử và

$$n(\Omega) = C_{15}^3 = \frac{15!}{3! \cdot 12!} = 455.$$

- Xét biến cố $A$: “Chọn được 3 học sinh chỉ thuộc hai khối”.

Sơ đồ hình cây biểu thị các khả năng thuận lợi cho biến cố $A$ (Hình 2).

Chọn 3 học sinh (HS) chỉ ở hai khối

Chọn 1 HS lớp 10: Có 5 cách chọn

Chọn 1 HS lớp 11: Có 5 cách chọn

Chọn 1 HS lớp 12: Có 5 cách chọn

Chọn 2 HS lớp 11: Có $C_5^2$ cách chọn

Chọn 2 HS lớp 12: Có $C_5^2$ cách chọn

Chọn 2 HS lớp 10: Có $C_5^2$ cách chọn

Chọn 2 HS lớp 12: Có $C_5^2$ cách chọn

Chọn 2 HS lớp 10: Có $C_5^2$ cách chọn

Chọn 2 HS lớp 11: Có $C_5^2$ cách 

,cer
15,0.583776


In [ ]:
def inspect_tables(sample_id: str):
    row = manifest[manifest['id'] == sample_id].iloc[0]
    ref_md = read_text(Path(row['gt_md_path']))
    pred_md = read_text(PRED_DIR / f'{sample_id}.md')
    ref_tables = extract_tables(ref_md)
    pred_tables = extract_tables(pred_md)
    matches, doc_teds = match_tables(ref_tables, pred_tables)

    print('sample_id:', sample_id)
    print('ref tables:', len(ref_tables), 'pred tables:', len(pred_tables), 'doc_teds:', round(doc_teds, 4))
    print('matches:', matches)

    for label, tables in [('REF', ref_tables), ('PRED', pred_tables)]:
        print('\n' + label)
        for idx, t in enumerate(tables):
            print(f'--- table {idx}; shape={len(t)}x{max([len(r) for r in t], default=0)}')
            display(pd.DataFrame(t))

# Example:
inspect_tables(ok.sort_values('table_teds_doc').iloc[2]['id'])


sample_id: vi_text_test3_page_003
ref tables: 1 pred tables: 2 doc_teds: 0.4934
matches: [(0, 0, 0.9867816091954023)]

REF
--- table 0; shape=7x3


,0,1,2
0,Nhóm,Tần số,Tần số tích lũy
1,[50 ; 60),4,4
2,[60 ; 70),5,9
3,[70 ; 80),23,32
4,[80 ; 90),6,38
5,[90 ; 100),2,40
6,,$n = 40$,



PRED
--- table 0; shape=7x3


,0,1,2
0,Nhóm,Tần số,Tần số tích luỹ
1,[50 ; 60),4,4
2,[60 ; 70),5,9
3,[70 ; 80),23,32
4,[80 ; 90),6,38
5,[90 ; 100),2,40
6,,n = 40,


--- table 1; shape=4x10


,0,1,2,3,4,5,6,7,8,9
0,100,105,115,116,130,135,138,132,135,120
1,125,128,120,124,140,140,146,145,142,142
2,145,148,150,150,159,155,151,156,155,151
3,154,152,153,160,162,175,176,165,188,198
